In [14]:
import time
import json
import torch
import joblib
import numpy as np
from pathlib import Path
from credit_risk.dataset import load_splits, AFTER_EDA
from credit_risk.features import prep_one_split, NUMERICAL_COLS
from credit_risk.modeling.mlp import MLP

In [2]:
cwd = Path.cwd()
project_root = cwd.parent
nn_path = project_root / 'models' / 'mlp'

In [3]:
# load the preprocessor
pp_path = project_root / 'data' / 'processed' / 'features' / 'preprocessor.pkl'
pp = joblib.load(pp_path)

### Latency

In [4]:
_, _, test_df, _ = load_splits(path=AFTER_EDA)

2026-07-05 16:06:36.190 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-07-05 16:06:36.200 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-07-05 16:06:36.484 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [5]:
with open(nn_path / 'architecture.json') as f:
    archi = json.load(f)

In [6]:
# loading the model
state_dict = torch.load(nn_path / 'model_state.pt')
mlp_model = MLP(input_dim=archi['input_dim'], hidden_dim=archi['hidden_dim'], dropout=archi['dropout'])
mlp_model.load_state_dict(state_dict=state_dict)

<All keys matched successfully>

In [17]:
pt_x_test, y_test = prep_one_split(test_df)

2026-07-05 20:16:01.210 | INFO     | credit_risk.features:prep_one_split:217 - Inside Function: prep_one_split
2026-07-05 20:16:01.210 | INFO     | credit_risk.features:sorting_with_issue_d:143 - Sorting the dataframe wrt to issue_d...
2026-07-05 20:16:01.379 | INFO     | credit_risk.features:sorting_with_issue_d:148 - Sorted successfully!
2026-07-05 20:16:01.379 | INFO     | credit_risk.features:split_target_and_features:153 - Inside Function: split_target_and_features
2026-07-05 20:16:01.379 | INFO     | credit_risk.features:split_target_and_features:154 - Splitting the target and the features...
2026-07-05 20:16:01.381 | INFO     | credit_risk.features:split_target_and_features:157 - features and target are splitted successfully...
2026-07-05 20:16:01.381 | INFO     | credit_risk.features:add_credit_yrs:170 - Inside Function: add_credit_yrs
2026-07-05 20:16:01.382 | INFO     | credit_risk.features:add_credit_yrs:172 - Adding the feature 'credit_yrs'
2026-07-05 20:16:01.386 | INFO   

In [ ]:
# MLP
# latency check
mlp_model.eval()
with torch.no_grad():
    # warm up
    for _ in range(10):
        X_test = pp.transform(pt_x_test[:1])
        single_tensor = torch.from_numpy(X_test.astype(np.float32))
        _ = torch.sigmoid(mlp_model(single_tensor))
        
    n_trials = 100
    times = []
    for _ in range(n_trials):
        start = time.perf_counter()
        X_test = pp.transform(pt_x_test[:1])
        single_tensor = torch.from_numpy(X_test.astype(np.float32))
        _ = torch.sigmoid(mlp_model(single_tensor))
        end = time.perf_counter()
        times.append((end - start) * 1000)
        
print(f"median: {np.median(times):.3f} ms")
print(f'P95: {np.percentile(times, 95):.3f} ms')

median: 1.208 ms
P95: 1.536 ms


In [ ]:
# mlp
# batch
with torch.no_grad():
    # warmup
    for _ in range(3):
        X_test = pp.transform(pt_x_test[:1000])
        single_batch = torch.from_numpy(X_test.astype(np.float32))
        _ = torch.sigmoid(mlp_model(single_batch))
        
    n_trials = 50
    times = []
    for _ in range(50):
        start = time.perf_counter()
        X_test = pp.transform(pt_x_test[:1000])
        single_batch = torch.from_numpy(X_test.astype(np.float32))
        _ = torch.sigmoid(mlp_model(single_batch))
        end = time.perf_counter()
        
        times.append((end - start) * 1000)
        
print(f"Median: {np.median(times):.3f} ms")
print(f"per-row: {(np.median(times)/1000):.3f} ms")

Median: 2.418 ms
per-row: 0.002 ms


### Size

In [13]:
nn_size = (nn_path / 'model_state.pt').stat().st_size / (1024**2)
pp_size = pp_path.stat().st_size / (1024**2)

print(f"Preprocessor for MLP size alone: {pp_size:.2f} MB")
print(f"MLP statedict size: {nn_size:.2f} MB")
print(f"MLP (pp + model): {(nn_size + pp_size):.2f} MB")

Preprocessor for MLP size alone: 0.01 MB
MLP statedict size: 0.11 MB
MLP (pp + model): 0.12 MB


### Perturbation test

In [33]:
np.random.seed(42)
perturb_test_set = pt_x_test.copy()

cols_to_purturb = [col for col in NUMERICAL_COLS if col in perturb_test_set.columns]

noise = np.random.normal(1, 0.1, size=(len(perturb_test_set), len(cols_to_purturb)))
perturb_test_set[cols_to_purturb] = perturb_test_set[cols_to_purturb].values * noise

In [34]:
with torch.no_grad():
    perturb_test_set = pp.transform(perturb_test_set)
    perturb_test_set = torch.from_numpy(perturb_test_set.astype(np.float32))
    perturb_test_pp = torch.sigmoid(mlp_model(perturb_test_set)).numpy()

In [35]:
from sklearn.metrics import average_precision_score

mlp_perturbed_pr = average_precision_score(y_true=y_test, y_score=perturb_test_pp)
mlp_clean_pr = 0.3038 
print(f"MLP clean:     {mlp_clean_pr:.4f}")
print(f"MLP perturbed: {mlp_perturbed_pr:.4f}")
print(f"Delta: {mlp_perturbed_pr - mlp_clean_pr:+.4f}")

MLP clean:     0.3038
MLP perturbed: 0.2833
Delta: -0.0205


### Directional Expectation Test

In [83]:
# DTI
change = 15
de_test_set = pt_x_test.sample(n=500, random_state=42)
non_de_test_set = de_test_set.copy()
de_test_set['dti'] += change

In [84]:
with torch.no_grad():
    de_test_set = pp.transform(de_test_set)
    de_test_set = torch.from_numpy(de_test_set.astype(np.float32))
    de_test_set_pp = torch.sigmoid(mlp_model(de_test_set)).numpy()
    
    non_de_test_set = pp.transform(non_de_test_set)
    non_de_test_set = torch.from_numpy(non_de_test_set.astype(np.float32))
    non_de_test_set_pp = torch.sigmoid(mlp_model(non_de_test_set)).numpy()

In [85]:
avg_delta_pp = (de_test_set_pp - non_de_test_set_pp).mean()

print(f"For MLP: added {change} in dti: Avg delta in predicted probs(changed_dti - unchanged_dti): {avg_delta_pp:.4f}")

For MLP: added 15 in dti: Avg delta in predicted probs(changed_dti - unchanged_dti): 0.0376


In [92]:
# annual_inc
change = 10000
de_test_set = pt_x_test.sample(n=500, random_state=42)
non_de_test_set = de_test_set.copy()
de_test_set['annual_inc'] += change

In [93]:
with torch.no_grad():
    de_test_set = pp.transform(de_test_set)
    de_test_set = torch.from_numpy(de_test_set.astype(np.float32))
    de_test_set_pp = torch.sigmoid(mlp_model(de_test_set)).numpy()
    
    non_de_test_set = pp.transform(non_de_test_set)
    non_de_test_set = torch.from_numpy(non_de_test_set.astype(np.float32))
    non_de_test_set_pp = torch.sigmoid(mlp_model(non_de_test_set)).numpy()

In [94]:
avg_delta_pp = (de_test_set_pp - non_de_test_set_pp).mean()

print(f"For MLP: added {change} in annual_inc: Avg delta in predicted probs(changed_dti - unchanged_dti): {avg_delta_pp:.6f}")

For MLP: added 10000 in annual_inc: Avg delta in predicted probs(changed_dti - unchanged_dti): -0.007140
